# 04. 모델링 (Modeling)

**모델 스택**: Logistic Regression (베이스라인) → Decision Tree → Random Forest → LightGBM → XGBoost  
**검증 방법**: Stratified K-Fold (k=5), SMOTE는 각 fold 내부에서 적용  
**주 평가 지표**: AUC-ROC  
**부 평가 지표**: F1-score, Precision, Recall

---
## 0. 라이브러리 & 설정

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score, roc_curve
)
import lightgbm as lgb
import xgboost as xgb

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8')
sns.set_theme(font_scale=1.5)
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False
os.makedirs('../outputs', exist_ok=True)
print('라이브러리 로드 완료')

라이브러리 로드 완료


---
## 1. 데이터 로드

In [2]:
X_train = pd.read_parquet('../data/interim/X_train.parquet')
y_train = pd.read_parquet('../data/interim/y_train.parquet')['label']
X_test  = pd.read_parquet('../data/interim/X_test.parquet')
y_test  = pd.read_parquet('../data/interim/y_test.parquet')['label']

feature_cols = X_train.columns.tolist()

print(f'X_train: {X_train.shape}  X_test: {X_test.shape}')
print(f'피처: {feature_cols}')
print(f'\nTrain 클래스 분포: {y_train.value_counts().to_dict()}')
print(f'Test  클래스 분포: {y_test.value_counts().to_dict()}')

X_train: (67496, 4)  X_test: (16875, 4)
피처: ['avg_delay', 'total_orders', 'delay_rate', 'recency_days']

Train 클래스 분포: {1: 67310, 0: 186}
Test  클래스 분포: {1: 16829, 0: 46}


---
## 2. CV 헬퍼 & SKF 설정

- SMOTE는 각 fold의 train split에만 적용 (pipeline 내부에서 처리)
- `predict_proba`가 없는 모델 대비 `decision_function` fallback 불필요 — 세 모델 모두 지원

In [3]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def run_cv(pipeline, X, y, model_name):
    """Stratified K-Fold CV. pipeline에 SMOTE 포함 가정."""
    results = {'auc': [], 'f1': [], 'precision': [], 'recall': []}
    print(f'=== {model_name} — CV (k=5) ===')
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        pipeline.fit(X_tr, y_tr)
        y_prob = pipeline.predict_proba(X_val)[:, 1]
        y_pred = pipeline.predict(X_val)
        results['auc'].append(roc_auc_score(y_val, y_prob))
        results['f1'].append(f1_score(y_val, y_pred, zero_division=0))
        results['precision'].append(precision_score(y_val, y_pred, zero_division=0))
        results['recall'].append(recall_score(y_val, y_pred, zero_division=0))
        print(f'  Fold {fold}: AUC={results["auc"][-1]:.4f}  F1={results["f1"][-1]:.4f}')
    print(f'  → CV AUC: {np.mean(results["auc"]):.4f} ± {np.std(results["auc"]):.4f}\n')
    return results


def eval_test(pipeline, X_test, y_test):
    """Test set 최종 평가."""
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    y_pred = pipeline.predict(X_test)
    return {
        'test_auc'      : roc_auc_score(y_test, y_prob),
        'test_f1'       : f1_score(y_test, y_pred, zero_division=0),
        'test_precision': precision_score(y_test, y_pred, zero_division=0),
        'test_recall'   : recall_score(y_test, y_pred, zero_division=0),
        'y_prob'        : y_prob,
    }

print('헬퍼 함수 정의 완료')

헬퍼 함수 정의 완료


---
## 3. Logistic Regression

- Pipeline: SMOTE → StandardScaler → LogisticRegression
- SMOTE로 불균형 처리

In [4]:
lr_pipe = ImbPipeline([
    ('smote',  SMOTE(random_state=42)),
    ('scaler', StandardScaler()),
    ('lr',     LogisticRegression(max_iter=1000, random_state=42)),
])

lr_cv = run_cv(lr_pipe, X_train, y_train, 'Logistic Regression')

=== Logistic Regression — CV (k=5) ===
  Fold 1: AUC=0.6343  F1=0.6946
  Fold 2: AUC=0.5651  F1=0.7108
  Fold 3: AUC=0.6536  F1=0.6930
  Fold 4: AUC=0.6476  F1=0.7060
  Fold 5: AUC=0.5865  F1=0.7057
  → CV AUC: 0.6174 ± 0.0352



In [5]:
# 전체 train으로 최종 학습 후 test 평가
lr_pipe.fit(X_train, y_train)
lr_test = eval_test(lr_pipe, X_test, y_test)

print('=== Logistic Regression — Test 결과 ===')
for k, v in lr_test.items():
    if k != 'y_prob':
        print(f'  {k}: {v:.4f}')

=== Logistic Regression — Test 결과 ===
  test_auc: 0.6220
  test_f1: 0.7011
  test_precision: 0.9981
  test_recall: 0.5403


---
## 4. LightGBM

In [6]:
lgb_pipe = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('lgb',   lgb.LGBMClassifier(n_estimators=300, random_state=42,
                                  n_jobs=-1, verbose=-1)),
])

lgb_cv = run_cv(lgb_pipe, X_train, y_train, 'LightGBM')

=== LightGBM — CV (k=5) ===
  Fold 1: AUC=0.5520  F1=0.9904
  Fold 2: AUC=0.4736  F1=0.9882
  Fold 3: AUC=0.4715  F1=0.9893
  Fold 4: AUC=0.5470  F1=0.9888
  Fold 5: AUC=0.6062  F1=0.9894
  → CV AUC: 0.5301 ± 0.0514



In [7]:
lgb_pipe.fit(X_train, y_train)
lgb_test = eval_test(lgb_pipe, X_test, y_test)

print('=== LightGBM — Test 결과 ===')
for k, v in lgb_test.items():
    if k != 'y_prob':
        print(f'  {k}: {v:.4f}')

=== LightGBM — Test 결과 ===
  test_auc: 0.5974
  test_f1: 0.9888
  test_precision: 0.9974
  test_recall: 0.9804


---
## 5. XGBoost

In [8]:
xgb_pipe = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('xgb',   xgb.XGBClassifier(n_estimators=300, random_state=42,
                                  n_jobs=-1, verbosity=0, eval_metric='auc')),
])

xgb_cv = run_cv(xgb_pipe, X_train, y_train, 'XGBoost')

=== XGBoost — CV (k=5) ===
  Fold 1: AUC=0.5571  F1=0.9812
  Fold 2: AUC=0.4982  F1=0.9798
  Fold 3: AUC=0.5415  F1=0.9820
  Fold 4: AUC=0.5030  F1=0.9810
  Fold 5: AUC=0.5698  F1=0.9822
  → CV AUC: 0.5339 ± 0.0287



In [9]:
xgb_pipe.fit(X_train, y_train)
xgb_test = eval_test(xgb_pipe, X_test, y_test)

print('=== XGBoost — Test 결과 ===')
for k, v in xgb_test.items():
    if k != 'y_prob':
        print(f'  {k}: {v:.4f}')

=== XGBoost — Test 결과 ===
  test_auc: 0.5504
  test_f1: 0.9799
  test_precision: 0.9974
  test_recall: 0.9631


---
## 6. Decision Tree

In [10]:
dt_pipe = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('dt',    DecisionTreeClassifier(random_state=42)),
])

dt_cv = run_cv(dt_pipe, X_train, y_train, 'Decision Tree')

=== Decision Tree — CV (k=5) ===
  Fold 1: AUC=0.5087  F1=0.9902
  Fold 2: AUC=0.4955  F1=0.9900
  Fold 3: AUC=0.5127  F1=0.9918
  Fold 4: AUC=0.5244  F1=0.9902
  Fold 5: AUC=0.4990  F1=0.9928
  → CV AUC: 0.5081 ± 0.0103



In [11]:
dt_pipe.fit(X_train, y_train)
dt_test = eval_test(dt_pipe, X_test, y_test)

print('=== Decision Tree — Test 결과 ===')
for k, v in dt_test.items():
    if k != 'y_prob':
        print(f'  {k}: {v:.4f}')

=== Decision Tree — Test 결과 ===
  test_auc: 0.5164
  test_f1: 0.9917
  test_precision: 0.9974
  test_recall: 0.9862


---
## 7. Random Forest

In [12]:
rf_pipe = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('rf',    RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)),
])

rf_cv = run_cv(rf_pipe, X_train, y_train, 'Random Forest')

=== Random Forest — CV (k=5) ===
  Fold 1: AUC=0.5115  F1=0.9911
  Fold 2: AUC=0.5046  F1=0.9908
  Fold 3: AUC=0.5440  F1=0.9918
  Fold 4: AUC=0.5293  F1=0.9908
  Fold 5: AUC=0.5531  F1=0.9927
  → CV AUC: 0.5285 ± 0.0185



In [13]:
rf_pipe.fit(X_train, y_train)
rf_test = eval_test(rf_pipe, X_test, y_test)

print('=== Random Forest — Test 결과 ===')
for k, v in rf_test.items():
    if k != 'y_prob':
        print(f'  {k}: {v:.4f}')

=== Random Forest — Test 결과 ===
  test_auc: 0.5578
  test_f1: 0.9915
  test_precision: 0.9974
  test_recall: 0.9858


---
## 8. 모델 비교

In [14]:
models = {
    'Logistic Regression': (lr_cv, lr_test),
    'Decision Tree'      : (dt_cv, dt_test),
    'Random Forest'      : (rf_cv, rf_test),
    'LightGBM'           : (lgb_cv, lgb_test),
    'XGBoost'            : (xgb_cv, xgb_test),
}

rows = []
for name, (cv, test) in models.items():
    rows.append({
        '모델'          : name,
        'CV AUC (mean)' : f"{np.mean(cv['auc']):.4f}",
        'CV AUC (std)'  : f"{np.std(cv['auc']):.4f}",
        'Test AUC'      : f"{test['test_auc']:.4f}",
        'Test F1'       : f"{test['test_f1']:.4f}",
        'Test Precision': f"{test['test_precision']:.4f}",
        'Test Recall'   : f"{test['test_recall']:.4f}",
    })

comp_df = pd.DataFrame(rows).set_index('모델')
print('=== 모델 비교 ===')
print(comp_df.to_string())

=== 모델 비교 ===
                    CV AUC (mean) CV AUC (std) Test AUC Test F1 Test Precision Test Recall
모델                                                                                        
Logistic Regression        0.6174       0.0352   0.6220  0.7011         0.9981      0.5403
Decision Tree              0.5081       0.0103   0.5164  0.9917         0.9974      0.9862
Random Forest              0.5285       0.0185   0.5578  0.9915         0.9974      0.9858
LightGBM                   0.5301       0.0514   0.5974  0.9888         0.9974      0.9804
XGBoost                    0.5339       0.0287   0.5504  0.9799         0.9974      0.9631


---
## 9. 결과 저장

In [15]:
# 모델 비교 테이블 저장
comp_df.to_csv('../outputs/model_comparison.csv')
print('[저장] ../outputs/model_comparison.csv')

print('\n=== 최종 모델 비교 ===')
print(comp_df.to_string())

[저장] ../outputs/model_comparison.csv

=== 최종 모델 비교 ===
                    CV AUC (mean) CV AUC (std) Test AUC Test F1 Test Precision Test Recall
모델                                                                                        
Logistic Regression        0.6174       0.0352   0.6220  0.7011         0.9981      0.5403
Decision Tree              0.5081       0.0103   0.5164  0.9917         0.9974      0.9862
Random Forest              0.5285       0.0185   0.5578  0.9915         0.9974      0.9858
LightGBM                   0.5301       0.0514   0.5974  0.9888         0.9974      0.9804
XGBoost                    0.5339       0.0287   0.5504  0.9799         0.9974      0.9631
